# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Ebrahim827/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

pd.set_option("display.width", 120)
RANDOM_SEED = 42

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
df["y_declining"] = (df["trend_direction"] == "down").astype(int)
print("Loaded:", df.shape[0], "rows x", df.shape[1], "cols — base rate declining:", round(df["y_declining"].mean(), 3))

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 219, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 219 (delta 90), reused 57 (delta 57), pack-reused 107 (from 2)
Receiving objects: 100% (219/219), 1.88 MiB | 8.67 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/flyrank-ml-internship-starter
Loaded: 30000 rows x 45 cols — base rate declining: 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Question shape is yes/no: is a page declining or not? Per the skill's method guide, that means start with Logistic Regression (readable), then compare against Random Forest (stronger, still explainable via permutation importance). Not using Gradient Boosting — no need to reach for more complexity than the comparison earns.

In [2]:
numeric_features = ["days_since_last_update", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "engaged_sessions_90d", "scroll_events_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct", "word_count", "content_age_days",
    "search_volume", "competition", "cpc", "days_with_impressions", "days_with_sessions"]
categorical_features = ["content_type", "main_intent", "competition_level"]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

# LEAKAGE GUARD: trend_direction / trend_pct / is_declining_label are the label (or derived from
# it) — never used as inputs. content_id / client_id are pseudonym IDs — grouping only, never features.
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['days_since_last_update', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'engaged_sessions_90d', 'scroll_events_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count', 'content_age_days', 'search_volume', 'competition', 'cpc', 'days_with_impressions', 'days_with_sessions']
Categorical features: ['content_type', 'main_intent', 'competition_level']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client_id, not random row-level. A random split would let the model learn client-specific quirks (writing style, industry, baseline traffic) and then "cheat" by seeing that same client again in the test set — inflating the score without proving anything about generalization. Not time-aware, since this snapshot is a fixed trailing-90-day window per row, not a time series.

In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

X = df[numeric_features + categorical_features]
y = df["y_declining"]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
df_test = df.iloc[test_idx]

overlap = set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"])
print(f"Train rows: {len(train_idx)}, Test rows: {len(test_idx)}")
print(f"Client overlap (should be empty): {overlap}")
print(f"Train base rate: {y_train.mean():.3f} | Test base rate: {y_test.mean():.3f}")

Train rows: 19166, Test rows: 10834
Client overlap (should be empty): set()
Train base rate: 0.532 | Test base rate: 0.559


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test split, same precision@K metric used for the Week-4 baseline. The baseline rule is recomputed here on this exact test slice (not just copied from last week's numbers).

In [4]:
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

logreg = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))])
logreg.fit(X_train, y_train)
proba_lr = logreg.predict_proba(X_test)[:, 1]

rf = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    k = min(k, len(order))
    return np.asarray(labels)[order[:k]].mean()

STALE_DAYS, VISIBLE_IMPRESSIONS = 180, 500
stale = (df_test["days_since_last_update"] >= STALE_DAYS).astype(int)
visible = (df_test["impressions_90d"] >= VISIBLE_IMPRESSIONS).astype(int)
baseline_score = (stale * visible * df_test["impressions_90d"]).values

base_rate = y_test.mean()
rows = []
for k in [10, 20, 50]:
    rows.append({
        "k": k,
        "base_rate": round(base_rate, 3),
        "baseline_precision@k": round(precision_at_k(baseline_score, y_test.values, k), 3),
        "logreg_precision@k": round(precision_at_k(proba_lr, y_test.values, k), 3),
        "random_forest_precision@k": round(precision_at_k(proba_rf, y_test.values, k), 3),
    })

comparison_table = pd.DataFrame(rows)
print("Random seed:", RANDOM_SEED)
print(comparison_table)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Random seed: 42
    k  base_rate  baseline_precision@k  logreg_precision@k  random_forest_precision@k
0  10      0.559                  0.80                0.40                       0.50
1  20      0.559                  0.65                0.40                       0.55
2  50      0.559                  0.64                0.46                       0.64


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1)
importances = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("Top 8 features by permutation importance:")
print(importances.head(8))

results = df_test.copy()
results["y_true"] = y_test.values
results["rf_proba"] = proba_rf
results["rf_pred"] = (results["rf_proba"] >= 0.5).astype(int)
wrong = results[results["y_true"] != results["rf_pred"]].copy()
wrong["confidence_gap"] = (wrong["rf_proba"] - 0.5).abs()
worst_wrong = wrong.sort_values("confidence_gap", ascending=False).head(3)

cols_to_show = ["content_id", "y_true", "rf_pred", "rf_proba",
                 "days_since_last_update", "impressions_90d", "avg_position", "ctr"]
print(worst_wrong[cols_to_show])

Top 8 features by permutation importance:
days_with_impressions    0.033819
content_age_days         0.014371
avg_position             0.011888
clicks_90d               0.007107
ctr                      0.004421
days_with_sessions       0.004338
sessions_90d             0.004052
impressions_90d          0.002538
dtype: float64
                 content_id  y_true  rf_pred  rf_proba  days_since_last_update  impressions_90d  avg_position   ctr
12364  content_3a4e24a3a6a8       1        0  0.036385                      20                1           4.0  0.00
26651  content_711c23a1d649       0        1  0.954536                      20             1538          11.9  0.26
17451  content_d015eb800625       0        1  0.942648                      20             1064           8.5  0.00


**What the model leans on**: the top feature is days_with_impressions (importance 0.034 — shuffling it costs the model about 3.4 percentage points of accuracy), followed by content_age_days (0.014) and avg_position (0.012). A cluster of traffic/engagement features (clicks_90d, ctr, days_with_sessions, sessions_90d, impressions_90d) trail behind at 0.002–0.007 — they matter, but far less. All of these plausibly relate to decline: fewer days with any impressions, older content, and weaker engagement are reasonable signs a page is losing traction. None of the importances are suspiciously large — if one feature dominated almost completely, that would be a leakage red flag. Here the signal is spread across several sensible features instead, which is a healthier sign.
Three wrong cases:

**content_3a4e24a3a6a8** — actually declining, but the model was confident it wasn't (3.6% predicted probability). It ranks well (position 4.0) but has only 1 impression and 0 clicks in 90 days — an unusual mismatch, since good rank almost always brings more visibility. The model likely over-trusted the strong position and missed that real traffic had already collapsed.

**content_711c23a1d649**— actually stable, but the model was 95.5% confident it was declining. Mid-pack position (11.9) with modest traffic (1,538 impressions) resembles a profile that trends downward elsewhere in the data, but this particular page held steady — the pattern didn't generalize to every page that fits it.

**content_d015eb800625** — actually stable, model 94.3% confident it was declining. Decent visibility (1,064 impressions, position 8.5) but 0% CTR — that combination likely reads as "failing" to the model, but a flat 0% CTR from the start isn't the same as declining; the model may be conflating "consistently weak" with "getting worse."

## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.